In [5]:
import pandas as pd

# ===== 1. Read CSV =====
df = pd.read_csv("results_all.csv")

# ===== 2. Filter by model and prompt_type =====
filtered_df = df[
    (df["model"] == "gemini-2.5-pro") &
    (df["prompt_type"] == "key_information")
].copy()

# ===== 3. Clean text fields =====
filtered_df["question"] = filtered_df["question"].astype(str).str.strip()
filtered_df["answer"] = filtered_df["answer"].astype(str).str.strip()

# Remove rows with missing or empty question/answer
filtered_df = filtered_df[
    (filtered_df["question"] != "") &
    (filtered_df["answer"] != "")
]

# ===== 4. Sort rows =====
filtered_df = filtered_df.sort_values(by=["source_type", "document"])

# ===== 5. Remove duplicate questions within each document =====
filtered_df = filtered_df.drop_duplicates(
    subset=["source_type", "document", "question"]
)

# ===== 6. Keep top 3 FAQs per document =====
filtered_df = (
    filtered_df
    .groupby(["source_type", "document"], as_index=False, group_keys=False)
    .head(3)
)

# ===== 7. Build Markdown content =====
md_lines = []
md_lines.append("# Smart FAQ Results\n")
md_lines.append(
    "This page shows representative FAQ results generated using **gemini-2.5-pro** with the **key_information** prompt.\n"
)

for (source_type, document), group in filtered_df.groupby(["source_type", "document"], sort=False):
    md_lines.append(f"## {document}")
    md_lines.append(f"**Source Type:** {source_type}\n")

    for i, (_, row) in enumerate(group.iterrows(), start=1):
        question = row["question"]
        answer = row["answer"]

        md_lines.append(f"### Q{i}. {question}")
        md_lines.append(f"{answer}\n")

    md_lines.append("---\n")

# ===== 8. Save Markdown file =====
output_file = "smart_faq_results1.md"
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

print(f"Markdown file saved as: {output_file}")

Markdown file saved as: smart_faq_results1.md
